# Logistic Regression

Imports and helpers

In [18]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib


## Model Training

## Model Evaluation

## Model Testing

## Saving the Model and Pipeline

## Insights

In [2]:
# Ensure directories exist
os.makedirs('../data/balanced100k', exist_ok=True)
os.makedirs('../data/models', exist_ok=True)
os.makedirs('../data/predictions', exist_ok=True)


In [4]:
# Load a sample of the raw training data (approx 100k rows)
raw_path = '../data/raw/train.csv'
print('Loading from', raw_path)
try:
    df = pd.read_csv(raw_path, nrows=200000)
except Exception:
    df = pd.read_csv(raw_path)
print('Initial rows read:', len(df))
if len(df) > 100000:
    df = df.sample(n=100000, random_state=42)
else:
    df = df.sample(frac=1, random_state=42)
print('Sampled rows:', len(df))


Loading from ../data/raw/train.csv
Initial rows read: 200000
Sampled rows: 100000


In [5]:
# Detect text and target columns heuristically
print('Columns:', list(df.columns))
text_candidates = [c for c in df.columns if df[c].dtype == 'object']
text_col = None

for c in text_candidates:
    if df[c].astype(str).str.len().median() > 20:
        text_col = c
        break

if text_col is None and text_candidates:
    text_col = text_candidates[0]

# Target heuristics
target_col = None
for candidate in ['label','sentiment','rating','score','stars','target']:
    if candidate in df.columns:
        target_col = candidate
        break

if target_col is None:
    # find a numeric column with small number of unique values
    for c in df.select_dtypes(include=[np.number]).columns:
        if df[c].nunique() < 20:
            target_col = c
            break
        
print('Detected text column:', text_col)
print('Detected target column:', target_col)


Columns: ['2', 'Stuning even for the non-gamer', 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^']
Detected text column: Stuning even for the non-gamer
Detected target column: 2


In [6]:
# If target is rating-like, convert to binary sentiment
if target_col is None:
    raise ValueError('Could not detect target column automatically. Inspect df.columns and set target_col manually.')
if str(df[target_col].dtype).startswith('int') or str(df[target_col].dtype).startswith('float'):
    uniq = df[target_col].unique()
    if len(uniq) > 2:
        # convert 1-5 ratings to binary: 4+ -> positive
        median_val = np.median(uniq)
        df['__y__'] = (df[target_col] >= (median_val+1)).astype(int)
    else:
        df['__y__'] = df[target_col].astype(int)
else:
    # assume categorical strings already representing labels
    df['__y__'] = df[target_col].astype('category').cat.codes
y_counts = df['__y__'].value_counts()
print('Class distribution in sampled data:\n', y_counts)


Class distribution in sampled data:
 __y__
2    50412
1    49588
Name: count, dtype: int64


In [7]:
# Create a balanced dataset and save it
n_classes = df['__y__'].nunique()
per_class = min(df['__y__'].value_counts().min(), 100000 // n_classes)
frames = []
for cls in sorted(df['__y__'].unique()):
    subset = df[df['__y__'] == cls]
    frames.append(subset.sample(n=per_class, random_state=42))
balanced = pd.concat(frames).sample(frac=1, random_state=42).reset_index(drop=True)
print('Balanced class counts:', balanced['__y__'].value_counts().to_dict())
balanced_path = 'data/balanced100k/balanced.csv'
balanced.to_csv(balanced_path, index=False)
print('Saved balanced dataset to', balanced_path)


Balanced class counts: {1: 49588, 2: 49588}
Saved balanced dataset to data/balanced100k/balanced.csv


In [8]:
# Load cleaned dataset (from preprocessing step) if available, else fall back to balanced dataframe
from pathlib import Path
cleaned_default = Path('..') / 'data' / 'processed' / 'cleaned_dataset.csv'
try:
    cp = cleaned_path  # variable from earlier preprocess cell
except NameError:
    cp = cleaned_default
if Path(str(cp)).exists():
    df_model = pd.read_csv(str(cp))
    clean_col = [c for c in df_model.columns if c.endswith('_clean')][0]
    print('Loaded cleaned dataset from', cp)
else:
    df_model = cleaned_df if 'cleaned_df' in globals() else balanced
    clean_col = clean_col if 'clean_col' in globals() else (text_col + '_clean' if text_col else text_col)
    print('Using in-memory dataframe with rows:', len(df_model))

# Train/test split and vectorization
X = df_model[clean_col].fillna('')
y = df_model['__y__']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train_t = vectorizer.fit_transform(X_train)
X_test_t = vectorizer.transform(X_test)

# Train Logistic Regression
clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train_t, y_train)
y_pred = clf.predict(X_test_t)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred))

# Save model, vectorizer, and predictions
joblib.dump(clf, 'data/models/logistic_regression.joblib')
joblib.dump(vectorizer, 'data/models/tfidf_vectorizer.joblib')
preds_df = X_test.to_frame(name=clean_col) if isinstance(X_test, pd.Series) else pd.DataFrame({clean_col: X_test})
preds_df['y_true'] = y_test.values
preds_df['y_pred'] = y_pred
preds_df.to_csv('data/predictions/logistic_predictions.csv', index=False)
print('Saved model and predictions')

/home/aang/miniconda3/envs/mlqueens/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Accuracy: 0.8294515023190159

Classification report:
               precision    recall  f1-score   support

           1       0.85      0.80      0.83      9918
           2       0.81      0.85      0.83      9918

    accuracy                           0.83     19836
   macro avg       0.83      0.83      0.83     19836
weighted avg       0.83      0.83      0.83     19836

Saved model and predictions
